In [ ]:
from collections import defaultdict

class TreeNode:
    def __init__(self, item, count):
        self.item = item
        self.count = count
        self.parent = None
        self.children = {}
        self.link = None

    def increment(self, count):
        self.count += count

def build_fp_tree(transactions, min_support):
    # Step 1: Calculate item frequencies
    item_counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            item_counts[item] += 1

    # Remove items that don't meet the minimum support
    item_counts = {item: count for item, count in item_counts.items() if count >= min_support}
    if not item_counts:
        return None, None

    # Step 2: Sort items by frequency
    sorted_items = sorted(item_counts, key=lambda item: (-item_counts[item], item))
    header_table = {item: None for item in sorted_items}

    # Step 3: Build the FP-Tree
    root = TreeNode(None, 0)
    for transaction in transactions:
        # Filter and sort transaction
        filtered_transaction = [item for item in transaction if item in item_counts]
        sorted_transaction = sorted(filtered_transaction, key=lambda item: (-item_counts[item], item))
        # Insert transaction into FP-Tree
        current_node = root
        for item in sorted_transaction:
            if item in current_node.children:
                current_node.children[item].increment(1)
            else:
                new_node = TreeNode(item, 1)
                new_node.parent = current_node
                current_node.children[item] = new_node
                # Update header table
                if header_table[item] is None:
                    header_table[item] = new_node
                else:
                    # Link nodes
                    node = header_table[item]
                    while node.link is not None:
                        node = node.link
                    node.link = new_node
            current_node = current_node.children[item]

    return root, header_table

def mine_fp_tree(header_table, min_support):
    def find_conditional_pattern_base(item):
        conditional_pattern_base = []
        node = header_table[item]
        while node is not None:
            path = []
            current_node = node.parent
            while current_node.item is not None:
                path.append(current_node.item)
                current_node = current_node.parent
            conditional_pattern_base.append((path, node.count))
            node = node.link
        return conditional_pattern_base

    def construct_conditional_fp_tree(conditional_pattern_base):
        conditional_tree = TreeNode(None, 0)
        item_counts = defaultdict(int)
        for path, count in conditional_pattern_base:
            for item in path:
                item_counts[item] += count

        item_counts = {item: count for item, count in item_counts.items() if count >= min_support}
        if not item_counts:
            return None

        header_table = {item: None for item in item_counts}
        for path, count in conditional_pattern_base:
            filtered_path = [item for item in path if item in item_counts]
            sorted_path = sorted(filtered_path, key=lambda item: (-item_counts[item], item))
            current_node = conditional_tree
            for item in sorted_path:
                if item in current_node.children:
                    current_node.children[item].increment(count)
                else:
                    new_node = TreeNode(item, count)
                    new_node.parent = current_node
                    current_node.children[item] = new_node
                    # Update header table
                    if header_table[item] is None:
                        header_table[item] = new_node
                    else:
                        node = header_table[item]
                        while node.link is not None:
                            node = node.link
                        node.link = new_node
                current_node = current_node.children[item]

        return conditional_tree, header_table

    def recursive_mine_tree(header_table, prefix):
      patterns = {}
      for item in header_table.keys():
        new_prefix = prefix + [item]
        patterns[tuple(new_prefix)] = sum(node.count for node in iterate_nodes(header_table[item]))
        conditional_pattern_base = find_conditional_pattern_base(item)
        result = construct_conditional_fp_tree(conditional_pattern_base) # Assign the result to a single variable
        if result: # Check if the result is not None
            conditional_tree, conditional_header_table = result # Unpack only if result is not None
            patterns.update(recursive_mine_tree(conditional_header_table, new_prefix))
      return patterns

    def iterate_nodes(node):
        while node:
            yield node
            node = node.link

    return recursive_mine_tree(header_table, [])

# Sample dataset
transactions = [
    ['A', 'B', 'D'],
    ['B', 'C'],
    ['A', 'B', 'C', 'E'],
    ['B', 'E'],
    ['A', 'B', 'C', 'E']
]

# Minimum support threshold
min_support = 3

# Step 1: Build the FP-Tree
fp_tree, header_table = build_fp_tree(transactions, min_support)
print("FP-Tree built.")

# Step 2: Mine Frequent Patterns
frequent_patterns = mine_fp_tree(header_table, min_support)
print("Frequent Patterns:")
for pattern, support in frequent_patterns.items():
    print(f"Pattern: {pattern}, Support: {support}")
